# Task 3 - Turn Text Into Numbers (One-Hot Encoding)

**The problem in one sentence:** a machine-learning model does *math* — it can only work with numbers. But three of our columns contain **text**: `protocol_type` (`tcp`/`udp`/`icmp`), `service` (`http`/`smtp`/…), and `flag` (`SF`/`S0`/…). The model can't multiply `"tcp"`. So before training, we must convert that text into numbers — without inventing fake relationships. That conversion is this task.

**Terms defined as we go** (also in `reference/CONCEPTS_GLOSSARY.md`):
- **Categorical feature** = a column whose values are *labels/categories*, not quantities. `protocol_type` is categorical: `tcp`, `udp`, `icmp` are names, not amounts.
- **Numeric feature** = a column of actual numbers you can do math on, like `src_bytes`.
- **One-hot encoding** = the standard way to turn a categorical column into numbers (defined in detail in Step 2).

## Step 0 — Reload the data
Fresh notebook, so rebuild `df` like before: import pandas, define `all_columns`, `read_csv`, add `is_attack` as a **0/1 int** (remember `.astype(int)` this time!).

In [ ]:
# TODO: reload the training data into df, with column names and a 0/1 is_attack column
import pandas as pd

feature_columns=[
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
]

all_columns=feature_columns+["label","difficulty"]

df=pd.read_csv("/Users/rohanb/06_projects/CYBERSECURITY/data/raw/KDDTrain+.txt", header=None, names=all_columns)
df

In [ ]:
df["is_attack"]=(df["label"]!="normal").astype(int)
df

## Step 1 - See the problem for yourself
Let's confirm which columns are text vs numbers.

**Term — `dtype`:** short for 'data type'. pandas labels each column with the *kind* of data it holds: `int64`/`float64` = numbers; text shows as `object` (or `str` in newer pandas) — either way it means **text**.

**Your job:** print `df.dtypes` and find the columns that are text. Then, to build intuition, print the unique values of `protocol_type`.

**Hints:**
- `df.dtypes` lists every column and its type.
- `df["protocol_type"].unique()` shows the distinct values it contains.
- You'll find **4** text columns: `protocol_type`, `service`, `flag` — the 3 categorical *features* we'll encode — **plus `label`**, which is text too but is the *answer* (we drop it later in Step 5, we never encode it). Only the 3 features get one-hot encoded.

In [ ]:
# TODO: print df.dtypes, and df["protocol_type"].unique()
df.dtypes

In [ ]:
df["protocol_type"].unique()

In [ ]:
df["service"].unique()

In [ ]:
df["flag"].unique()

## Step 2 - Understand one-hot encoding (read this before coding)

**Why not just map tcp→1, udp→2, icmp→3?** Because that invents a *fake order/relationship*: it would tell the model `icmp (3) > udp (2) > tcp (1)`, and that `udp` is 'between' the other two. That's nonsense — they're just names, not ranks. A model would wrongly do math on those numbers.

**One-hot encoding** fixes this. It replaces one categorical column with **several new 0/1 columns — one per category** — where exactly one is 'hot' (=1) per row. Example for `protocol_type`:

| original | → | protocol_type_tcp | protocol_type_udp | protocol_type_icmp |
|---|---|---|---|---|
| tcp  | → | **1** | 0 | 0 |
| udp  | → | 0 | **1** | 0 |
| icmp | → | 0 | 0 | **1** |

Now each category is a separate yes/no (1/0) column with no fake ordering — pure numbers the model can safely use. ('One-hot' = one column is 'hot'/1, the rest are 0.)

Because `service` has ~70 possible values, one-hotting it creates ~70 columns — that's why our column count will jump a lot. That's expected and fine.

## Step 3 - Apply one-hot encoding
pandas has a built-in for this: **`pd.get_dummies`** ('dummy variables' is the statistics name for these 0/1 columns).

**Your job:** create a new DataFrame `df_encoded` where the 3 text columns are one-hot encoded, but all the numeric columns stay as they are.

**Hints:**
- `pd.get_dummies(df, columns=[...])` one-hot-encodes *only* the columns you list and leaves the rest untouched.
- Pass the 3 categorical column names in that `columns=` list.
- Assign the result to `df_encoded`.
- Optional: add `dtype=int` so the new columns are `0/1` ints instead of `True/False`.

In [ ]:
# TODO: df_encoded = pd.get_dummies(df, columns=[...])
df_encoded=pd.get_dummies(df, columns=["protocol_type","service","flag"],dtype=int)
df_encoded

## Step 4 - Confirm the transformation
Check what changed.

**Your job:** compare `df.shape` (before) with `df_encoded.shape` (after), and peek at the new columns.

**Hints:**
- Print both shapes. Columns should jump from 44 to well over 100 (each category became its own column).
- `df_encoded.columns.tolist()` (or `df_encoded.head()`) lets you eyeball the new `protocol_type_tcp`, `service_http`, `flag_SF`, ... columns.
- Sanity check: `df_encoded["protocol_type_tcp"].sum()` should equal the number of tcp rows you saw back in Task 2 (102,689).

In [ ]:
# TODO: print df.shape and df_encoded.shape; inspect the new columns
df.shape

In [ ]:
df_encoded.shape

In [ ]:
df_encoded["protocol_type_tcp"].sum()

## Step 5 - Separate the clues from the answer (build X and y)
One more prep step. In ML we conventionally split the data into two pieces:
- **`X`** = the **features** (all the input clues the model looks at). Capital X by convention.
- **`y`** = the **target/label** (the single answer we want it to predict). Lowercase y by convention.

The model learns the mapping `X → y`. Crucially, `X` must NOT contain the answer or anything that gives it away, or the model 'cheats'.

**Your job:**
1. Make `y = df_encoded["is_attack"]`.
2. Make `X` = everything EXCEPT the columns that are the answer or leak it: drop `is_attack`, `label` (the attack name — that's the answer in words!), and `difficulty` (not a real network feature).

**Hints:**
- Drop columns with `df_encoded.drop(columns=[...])`.
- Wait — is `label` still a column after Step 3? You didn't one-hot it, so yes, it's still there as text. It must be dropped from `X` (it literally *is* the answer).
- Print `X.shape` and `y.shape`. `X` should have the same number of rows as `y`, and a few fewer columns than `df_encoded`.

In [ ]:
# TODO: build y (the target) and X (the features, with answer-columns dropped)
X= df_encoded.drop(columns=["is_attack", "label","difficulty"])
y=df_encoded["is_attack"]
print(X.shape)
print(y.shape)

### ✅ You pass Task 3 when:
1. You found the 3 text (`object`) columns and can say *why* text is a problem for a model.
2. `df_encoded` has 100+ columns, all numeric (no more `object` dtype among the features).
3. You can explain, in one sentence, why one-hot beats mapping tcp→1/udp→2/icmp→3.
4. `X` (features) and `y` (target) exist, `X` has no `label`/`is_attack`/`difficulty`, and `X.shape[0] == y.shape[0]`.

Paste me your shapes + your one-sentence answer to #3, and I'll review — then Task 4 is training the model on `X` and `y`.